In [ ]:
!pip install -U transformers accelerate safetensors

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# --------------------------
# 1️⃣  Set your model path
# --------------------------
MODEL_PATH = "Qwen/Qwen2-0.5B"   # change this if your model folder has a different name

# --------------------------
# 2️⃣  Load model and tokenizer (robust to CPU/GPU)
# --------------------------
print("🔹 Loading your LegalMate model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Load model with sensible defaults for GPU/CPU
try:
    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )
        model.to("cpu")
except Exception as e:
    # fallback: try without trust_remote_code (if model is local and standard)
    print("⚠️ Primary load failed, retrying without trust_remote_code:", e)
    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float32, low_cpu_mem_usage=True)
        model.to("cpu")

# --------------------------
# 3️⃣  Start simple chat loop
# --------------------------
print("\n💬 LegalMate Chatbot is ready! (type 'exit' to quit)\n")

while True:
    query = input("👤 You: ").strip()
    if query.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break

    inputs = tokenizer(query, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # remove repeated user text from start if needed
    if response.startswith(query):
        response = response[len(query):].strip()

    print(f"🤖 LegalMate: {response}\n")

🔹 Loading your LegalMate model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


💬 LegalMate Chatbot is ready! (type 'exit' to quit)

👤 You: kya apko roman urdu ati hai?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🤖 LegalMate: (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman urdu ati hai? (1999)
Kya apko roman

👤 You: exit
👋 Goodbye!
